In [2]:
import sqlite3
import pandas as pd

In [3]:
conn = sqlite3.connect("../data/flipkart_smartphones.db")

In [3]:
query = """
SELECT * FROM price_history"""
pd.read_sql_query(query, conn)


,product_name,brand,current_price,original_mrp,discount_pct,rating,ratings_count,review_count,scraped_at,page_number,scraped_date
0,"vivo T5x 5G (Star Silver, 128 GB)",Vivo,22999,28999.0,20.0,4.4,12342,703,2026-06-30 16:02:20,1,2026-06-30
1,"vivo T5x 5G (Cyber Green, 128 GB)",Vivo,22999,28999.0,20.0,4.4,12342,703,2026-06-30 16:02:20,1,2026-06-30
2,"realme P4 Lite 5G (Mosaic Green, 64 GB)",Realme,15499,28999.0,46.0,4.3,8068,414,2026-06-30 16:02:20,1,2026-06-30
3,"vivo T5x 5G (Cyber Green, 256 GB)",Vivo,26999,33999.0,20.0,4.4,14728,981,2026-06-30 16:02:20,1,2026-06-30
4,"vivo T5x 5G (Cyber Green, 128 GB)",Vivo,24999,30999.0,19.0,4.4,14728,981,2026-06-30 16:02:20,1,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...
1336,"MOTOROLA Edge 70 Fusion (Pantone SILHOUETTE, 2...",Motorola,31999,45999.0,30.0,4.4,12406,1181,2026-07-06 16:54:24,8,2026-07-06
1337,"OnePlus Nord CE6 (Pitch Black, 128 GB)",OnePlus,33412,40999.0,18.0,4.5,302,17,2026-07-06 16:54:24,8,2026-07-06
1338,"Samsung S25 Ultra 5G (Titanium Gray, 256 GB)",Samsung,89825,129999.0,30.0,4.6,558,43,2026-07-06 16:54:24,8,2026-07-06
1339,"MOTOROLA Edge 70 Fusion (Pantone ORIENT BLUE, ...",Motorola,33999,49999.0,32.0,4.3,2450,226,2026-07-06 16:54:24,8,2026-07-06


In [4]:
query = """
SELECT * FROM products"""
pd.read_sql_query(query, conn)


,product_name,brand
0,"vivo T5x 5G (Star Silver, 128 GB)",Vivo
1,"vivo T5x 5G (Cyber Green, 128 GB)",Vivo
2,"realme P4 Lite 5G (Mosaic Green, 64 GB)",Realme
3,"vivo T5x 5G (Cyber Green, 256 GB)",Vivo
4,"vivo T5 Pro 5G (Glacier Blue, 128 GB)",Vivo
...,...,...
299,"MOTOROLA Edge 60 Pro (Pantone Dazzling Blue, 5...",Motorola
300,"Samsung Galaxy S25 FE 5G (Jetblack, 512 GB)",Samsung
301,"Samsung Galaxy S25 Plus 5G (Navy, 256 GB)",Samsung
302,"MOTOROLA Edge 70 Fusion (Pantone ORIENT BLUE, ...",Motorola


## Query 1 — Brand Overview: Price Positioning & Discount Strategy

Key findings:
- Apple have the pricing at ₹83,751 avg with minimal discount (7.1%)
- Realme offers the highest discounts (39.3%), suggesting an inflated MRP strategy
- Motorola dominates product count (49 variants)
- Google positions itself as premium (₹52,499 avg) with only 5 products
- iQOO follows a low-discount strategy like Apple, despite mid-range pricing

In [5]:
query = """
SELECT brand, 
       COUNT(DISTINCT product_name) AS products,
       ROUND(AVG(current_price), 0) AS avg_price,
       ROUND(AVG(discount_pct), 1) AS avg_discount
FROM price_history
GROUP BY brand
ORDER BY avg_price DESC
"""
pd.read_sql_query(query, conn)

,brand,products,avg_price,avg_discount
0,Apple,18,83751.0,7.1
1,Google,5,52499.0,6.0
2,iQOO,9,32898.0,5.9
3,OnePlus,12,31390.0,17.3
4,Samsung,45,25766.0,20.0
5,OPPO,26,24860.0,20.5
6,Motorola,49,24680.0,27.6
7,Vivo,22,24509.0,24.0
8,Infinix,4,21463.0,13.3
9,Redmi,15,20566.0,26.0


## Query 2 - Brand Ratings vs Price

Key findings:

- Apple have the highest rating despite low discount
- HMD (Nokia) have the 3rd highest rating (4.48) at budget price (₹17,726)
- Samsung ranks 11th in rating despite being a popular brand (45 products)
- POCO have the least rating even after giving decent discount

In [6]:
query = """
SELECT brand,
       COUNT(DISTINCT product_name) AS products,
       ROUND(AVG(current_price), 0) AS avg_price,
       ROUND(AVG(rating), 2) AS avg_rating,
       ROUND(AVG(discount_pct), 1) AS avg_discount       
FROM price_history
GROUP BY brand
ORDER BY avg_rating DESC
"""
pd.read_sql_query(query, conn)

,brand,products,avg_price,avg_rating,avg_discount
0,Apple,18,83751.0,4.64,7.1
1,Google,5,52499.0,4.50,6.0
2,HMD,8,17726.0,4.48,20.1
3,iQOO,9,32898.0,4.41,5.9
4,Vivo,22,24509.0,4.40,24.0
5,OPPO,26,24860.0,4.40,20.5
6,OnePlus,12,31390.0,4.39,17.3
7,Infinix,4,21463.0,4.38,13.3
8,Realme,33,20137.0,4.33,39.3
9,Redmi,15,20566.0,4.30,26.0


## Query 3 — Price Segment Analysis

Key findings:
- Mid-range (₹10K-20K) dominates with 129 products — 
  most competitive segment in Indian market
- Mid-range also has highest avg discount (31.2%) — 
  brands aggressively competing for volume buyers
- Premium segment (40K+) has highest rating (4.60) — 
  price and satisfaction correlate positively
- Budget under ₹10K is nearly extinct — only 20 products, 
  true budget market has shifted to ₹10K-15K range

In [7]:
query = """
SELECT 
    CASE 
        WHEN current_price < 10000 THEN 'Budget (under 10K)'
        WHEN current_price < 20000 THEN 'Mid-range (10K-20K)'
        WHEN current_price < 40000 THEN 'Upper mid (20K-40K)'
        ELSE 'Premium (above 40K)'
    END AS price_segment,
    COUNT(DISTINCT product_name) AS products,
    ROUND(AVG(current_price), 0) AS avg_price,
    ROUND(AVG(discount_pct), 1) AS avg_discount,
    ROUND(AVG(rating), 2) AS avg_rating
FROM price_history
GROUP BY 
    CASE 
        WHEN current_price < 10000 THEN 'Budget (under 10K)'
        WHEN current_price < 20000 THEN 'Mid-range (10K-20K)'
        WHEN current_price < 40000 THEN 'Upper mid (20K-40K)'
        ELSE 'Premium (above 40K)'
    END
ORDER BY avg_price ASC
"""
pd.read_sql_query(query, conn)

,price_segment,products,avg_price,avg_discount,avg_rating
0,Budget (under 10K),20,8361.0,21.8,4.13
1,Mid-range (10K-20K),129,15753.0,31.2,4.25
2,Upper mid (20K-40K),122,27822.0,23.0,4.37
3,Premium (above 40K),50,70599.0,13.6,4.60


## Query 4 — Top Rated Phones (min 1000 ratings)

Key findings:
- Top 10 highest rated phones with significant reviews 
  are Apple iPhones
- iPhone 17 series leads on rating (4.8) but 
  low review count suggests recent launch
- Confirms premium price = premium satisfaction 
- No Android phone breaks into top 10 when filtered 
  by minimum review threshold
- iPhone 15 dominates review count (246,757) suggesting older model but highest consumer engagement

In [8]:
query = """
SELECT product_name,
       brand,
       MIN(current_price) AS current_price,
       rating,
       MAX(ratings_count) AS ratings_count
FROM price_history
WHERE ratings_count > 1000
GROUP BY product_name
ORDER BY rating DESC, ratings_count DESC
LIMIT 10
"""
pd.read_sql_query(query, conn)

,product_name,brand,current_price,rating,ratings_count
0,"Apple iPhone 17 Pro Max (Silver, 256 GB)",Apple,137900,4.8,1424
1,"Apple iPhone 17 Pro Max (Cosmic Orange, 256 GB)",Apple,137900,4.8,1407
2,"Apple iPhone 17 Pro (Cosmic Orange, 256 GB)",Apple,122900,4.7,2889
3,"Apple iPhone 17 Pro (Deep Blue, 256 GB)",Apple,122900,4.7,2771
4,"Apple iPhone 17 Pro (Silver, 512 GB)",Apple,142900,4.7,2714
5,"Apple iPhone 17 Pro (Silver, 256 GB)",Apple,127900,4.7,2678
6,"Apple iPhone 15 (Black, 128 GB)",Apple,53900,4.6,246757
7,"Apple iPhone 16 (Black, 128 GB)",Apple,62900,4.6,195209
8,"Apple iPhone 16 (Ultramarine, 128 GB)",Apple,62900,4.6,195155
9,"Apple iPhone 16 (Pink, 128 GB)",Apple,62900,4.6,195047


## Query 5 — Price Drop Detection (First vs Last Day)

Key findings:

- products showed measurable price drops in just 7 days
- MOTOROLA g37 power has biggest drop — ₹6,000 (27.3%) in 7 days
- Motorola appears most frequently and most price-volatile brand
- Apple iPhone 15 dropped ₹2,500 (4.2%) shows that even premium 
  phones show movement

In [9]:
query = """
SELECT 
    e1.product_name,
    e1.brand,
    e1.current_price AS first_price,
    e2.current_price AS last_price,
    e1.current_price - e2.current_price AS price_drop,
    ROUND((e1.current_price - e2.current_price) * 100.0 / e1.current_price, 1) AS drop_pct
FROM price_history e1
JOIN price_history e2 
    ON e1.product_name = e2.product_name
WHERE e1.scraped_date = (SELECT MIN(scraped_date) FROM price_history)
  AND e2.scraped_date = (SELECT MAX(scraped_date) FROM price_history)
  AND e2.current_price < e1.current_price
GROUP BY e1.product_name
ORDER BY price_drop DESC
LIMIT 15
"""
pd.read_sql_query(query, conn)

,product_name,brand,first_price,last_price,price_drop,drop_pct
0,"MOTOROLA g37 power (PANTONE Nautical Blue, 128...",Motorola,21999,15999,6000,27.3
1,"MOTOROLA Edge 60 Pro (Pantone Shadow, 256 GB)",Motorola,33999,29999,4000,11.8
2,MOTOROLA Edge 60 Fusion 5G (PANTONE Mykonos Bl...,Motorola,27999,24999,3000,10.7
3,"Apple iPhone 15 (Black, 128 GB)",Apple,59400,56900,2500,4.2
4,"vivo T5x 5G (Cyber Green, 128 GB)",Vivo,24999,22999,2000,8.0
5,"OPPO K14x 5G (Prism Violet, 128 GB)",OPPO,19999,17999,2000,10.0
6,"MOTOROLA Edge 70 (PANTONE Bronze Green, 256 GB)",Motorola,29999,27999,2000,6.7
7,"Ai+ Pulse 2 (Purple, 64 GB)",Ai+,10499,8999,1500,14.3
8,"Ai+ Pulse 2 (Blue, 64 GB)",Ai+,10499,8999,1500,14.3
9,"Samsung Galaxy F06 5G (Lit Violet, 128 GB)",Samsung,13500,12499,1001,7.4


## Query 6 — MRP Inflation Detection

FINDING: Flipkart manipulates "original MRP" mid-sale period
to artificially inflate displayed discount percentages

Evidence caught during 7-day tracking period:
- realme P4x 5G: MRP inflated by ₹17,000 (from 21,999 to 38,999)
  while selling price barely changed — discount jumped 
  from ~2% to 35% artificially
- REDMI 15 5G: MRP inflated ₹13,000, real discount was 
  ~6% but displayed as 24%
- 55 products caught with MRP changes in just 7 days

This confirms the "big billion day" pricing strategy where 
Brands inflate MRP weeks before sales to make discounts 
look larger than they actually are

In [4]:
query = """
SELECT 
    product_name,
    brand,
    MIN(original_mrp) AS mrp_first,
    MAX(original_mrp) AS mrp_last,
    MAX(original_mrp) - MIN(original_mrp) AS mrp_change,
    MIN(current_price) AS lowest_price,
    MAX(current_price) AS highest_price,
    ROUND(AVG(discount_pct), 1) AS avg_discount
FROM price_history
WHERE original_mrp IS NOT NULL
GROUP BY product_name
HAVING MAX(original_mrp) != MIN(original_mrp)
ORDER BY mrp_change DESC
"""
pd.read_sql_query(query, conn)


,product_name,brand,mrp_first,mrp_last,mrp_change,lowest_price,highest_price,avg_discount
0,"realme P4x 5G (Elegant Pink, 128 GB)",Realme,21999.0,38999.0,17000.0,21499,22999,35.4
1,"realme P4x 5G (Lake Green, 128 GB)",Realme,23499.0,38999.0,15500.0,21999,22999,20.0
2,"REDMI 15 5G (Midnight Black, 128 GB)",Redmi,19999.0,32999.0,13000.0,18649,19994,24.4
3,"REDMI 15C 5G (Moonlight Blue, 128 GB)",Redmi,20999.0,31999.0,11000.0,17998,19358,23.4
4,"REDMI 15C 5G (Dusk Purple, 128 GB)",Redmi,20999.0,31999.0,11000.0,17998,19480,27.1
5,"REDMI 15 5G (Midnight Black, 256 GB)",Redmi,26999.0,36999.0,10000.0,23790,24499,22.0
6,"Samsung Galaxy M17 5G (Sapphire Black, 128 GB)",Samsung,16499.0,23999.0,7500.0,14960,17765,13.3
7,"MOTOROLA g37 power (PANTONE Nautical Blue, 128...",Motorola,30499.0,37999.0,7500.0,15999,21999,44.8
8,"realme P4R 5G (Titanium Glare, 128 GB)",Realme,34999.0,39999.0,5000.0,18999,20999,45.2
9,"realme P4R 5G (Silver Glare, 128 GB)",Realme,34999.0,39999.0,5000.0,18999,20999,45.4


## Query 7 — Most Popular Product Per Brand

Key findings:
- Apple iPhone 15 leads all brands with 246,757 ratings
- Motorola g45 5G and POCO C51 close behind — 
  budget segment drives volume in India
- Samsung's top product (Galaxy A14) has 3x fewer ratings 
  than Motorola's top despite more total variants
- Google and OnePlus are niche players — under 3,000 ratings 
  on their most popular phone

In [11]:
query = """
SELECT e1.product_name,
       e1.brand,
       e1.rating,
       MAX(e1.ratings_count) AS ratings_count,
       MAX(e1.review_count) AS review_count
FROM price_history e1
WHERE e1.ratings_count = (
    SELECT MAX(e2.ratings_count) 
    FROM price_history e2 
    WHERE e2.brand = e1.brand
)
GROUP BY e1.brand
ORDER BY ratings_count DESC
"""
pd.read_sql_query(query, conn)

,product_name,brand,rating,ratings_count,review_count
0,"Apple iPhone 15 (Black, 128 GB)",Apple,4.6,246757,9285
1,"Motorola g45 5G (Viva Magenta, 128 GB)",Motorola,4.3,230230,12038
2,POCO C51 - Locked with Airtel Prepaid (Power B...,POCO,4.0,226832,12033
3,"vivo T4x 5G (Glacial Teal, 128 GB)",Vivo,4.4,176790,8834
4,"Samsung Galaxy A14 5G (Light Green, 128 GB)",Samsung,4.2,82461,3915
5,OPPO K13 5G with 7000mAh and 80W SUPERVOOC Cha...,OPPO,4.4,75437,5032
6,"Ai+ Pulse 1 (Purple, 64 GB)",Ai+,4.2,43248,2833
7,"realme C61 (Safari Green, 64 GB)",Realme,4.3,37982,1428
8,"IQOO Z10R 5G (Aquamarine, 256 GB)",iQOO,4.4,5786,335
9,"REDMI 15 5G (Sandy Purple, 256 GB)",Redmi,4.3,4556,224


## Query 8 — Price Volatility Analysis

Key findings:
- OnePlus Nord CE6 Lite: changed price 16 times in 7 days — 
  highest frequency of any tracked phone
- OnePlus appears 6/15 times that means most price-volatile brand
- Budget and mid-range phones show higher volatility 
  than premium phones
- High volatility = opportunity for price-alert systems
  to notify buyers of dips

In [7]:
query = """
SELECT 
    product_name,
    brand,
    COUNT(DISTINCT current_price) AS price_changes,
    MIN(current_price) AS lowest_price,
    MAX(current_price) AS highest_price,
    MAX(current_price) - MIN(current_price) AS price_range
FROM price_history
GROUP BY product_name
HAVING COUNT(DISTINCT current_price) > 1
ORDER BY price_changes DESC
LIMIT 15
"""
pd.read_sql_query(query, conn)

,product_name,brand,price_changes,lowest_price,highest_price,price_range
0,"OnePlus Nord CE6 Lite (Hyper Black, 128 GB)",OnePlus,16,24515,27393,2878
1,"Samsung M06 5G (Sage Green, 128 GB)",Samsung,12,11829,16477,4648
2,"OnePlus Nord CE6 Lite (Vivid Mint, 128 GB)",OnePlus,10,24631,28242,3611
3,"OnePlus Nord CE6 (Pitch Black, 128 GB)",OnePlus,8,31645,34499,2854
4,"REDMI 15C 5G (Dusk Purple, 128 GB)",Redmi,7,17998,19480,1482
5,"OnePlus Nord CE6 (Fresh Blue, 256 GB)",OnePlus,7,36420,37800,1380
6,"OnePlus Nord CE6 (Fresh Blue, 128 GB)",OnePlus,7,32999,34380,1381
7,"Samsung Galaxy F06 5G (Lit Violet, 128 GB)",Samsung,6,11999,15499,3500
8,"Samsung Galaxy F06 5G (Bahama Blue, 128 GB)",Samsung,6,11500,15999,4499
9,"realme 15 Pro 5G (Silk Purple, 256 GB)",Realme,5,26999,30999,4000


In [4]:

query_drops = """
SELECT 
    e1.product_name,
    e1.brand,
    e1.current_price AS first_price,
    e2.current_price AS last_price,
    e1.current_price - e2.current_price AS price_drop,
    ROUND((e1.current_price - e2.current_price) * 100.0 / e1.current_price, 1) AS drop_pct
FROM price_history e1
JOIN price_history e2 
    ON e1.product_name = e2.product_name
WHERE e1.scraped_date = (SELECT MIN(scraped_date) FROM price_history)
  AND e2.scraped_date = (SELECT MAX(scraped_date) FROM price_history)
  AND e2.current_price < e1.current_price
GROUP BY e1.product_name
ORDER BY price_drop DESC
"""

df_drops = pd.read_sql_query(query_drops, conn)
df_drops.to_csv(
    r"C:\Users\91935\PycharmProjects\flipkart-smartphone-analysis\data\processed\price_drops.csv",
    index=False
)
print(f"Saved {len(df_drops)} products with price drops")
df_drops.head()

Saved 19 products with price drops


,product_name,brand,first_price,last_price,price_drop,drop_pct
0,"MOTOROLA g37 power (PANTONE Nautical Blue, 128...",Motorola,21999,15999,6000,27.3
1,"MOTOROLA Edge 60 Pro (Pantone Shadow, 256 GB)",Motorola,33999,29999,4000,11.8
2,MOTOROLA Edge 60 Fusion 5G (PANTONE Mykonos Bl...,Motorola,27999,24999,3000,10.7
3,"Apple iPhone 15 (Black, 128 GB)",Apple,59400,56900,2500,4.2
4,"vivo T5x 5G (Cyber Green, 128 GB)",Vivo,24999,22999,2000,8.0


In [5]:
# Export volatility data for Power BI Page 3
query_vol = """
SELECT 
    product_name,
    brand,
    COUNT(DISTINCT current_price) AS price_changes,
    MIN(current_price) AS lowest_price,
    MAX(current_price) AS highest_price,
    MAX(current_price) - MIN(current_price) AS price_range,
    ROUND(
        (MAX(current_price) - MIN(current_price)) * 100.0 / MIN(current_price)
    , 1) AS volatility_pct
FROM price_history
GROUP BY product_name
HAVING COUNT(DISTINCT current_price) > 1
ORDER BY price_changes DESC
"""

df_vol = pd.read_sql_query(query_vol, conn)
df_vol.to_csv(
    r"C:\Users\91935\PycharmProjects\flipkart-smartphone-analysis\data\processed\price_volatility.csv",
    index=False
)
print(f"Saved {len(df_vol)} volatile products")
df_vol.head()

Saved 126 volatile products


,product_name,brand,price_changes,lowest_price,highest_price,price_range,volatility_pct
0,"OnePlus Nord CE6 Lite (Hyper Black, 128 GB)",OnePlus,16,24515,27393,2878,11.7
1,"Samsung M06 5G (Sage Green, 128 GB)",Samsung,12,11829,16477,4648,39.3
2,"OnePlus Nord CE6 Lite (Vivid Mint, 128 GB)",OnePlus,10,24631,28242,3611,14.7
3,"OnePlus Nord CE6 (Pitch Black, 128 GB)",OnePlus,8,31645,34499,2854,9.0
4,"REDMI 15C 5G (Dusk Purple, 128 GB)",Redmi,7,17998,19480,1482,8.2
